In [11]:
import faiss
import numpy as np

import pickle

In [13]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [14]:
with open(
    "metadata_tokenizer_v1.pkl",
    "rb"
) as f:
    tokenizer=pickle.load(f)

In [15]:
encoder=load_model(
    "metadata_embedding_encoder_v1.keras"
)

In [4]:
metadata_embeddings = np.load(
    "metadata_embeddings.npy"
)

metadata_texts = np.load(
    "metadata_texts.npy"
)

metadata_filenames = np.load(
    "metadata_image_filenames.npy",
    allow_pickle=True
)

In [6]:
metadata_embeddings=metadata_embeddings.astype("float32")  #faiss requires float32 vectors.

In [7]:
#normalize embeddings
faiss.normalize_L2(metadata_embeddings)

In [9]:
#create faiss index
dimension=metadata_embeddings.shape[1]

index=faiss.IndexFlatIP(      #IP -> inner product.
    dimension
)

In [10]:
#add embeddings to index
index.add(metadata_embeddings)

In [16]:
max_length=24

#faiss retrieval func.
def retrieve_metadata_faiss(query_metadata, top_k=5):
    query_sequences=tokenizer.texts_to_sequences(
        [query_metadata]
    )

    query_padded=pad_sequences(
        query_sequences,
        maxlen=max_length,
        padding='post'
    )

    query_embedding=encoder.predict(
        query_padded
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    similarities, indices=index.search(
        query_embedding,
        top_k
    )

    results=[]

    for sim, idx in zip(similarities[0], indices[0]):
        results.append({
            "metadata":
            str(metadata_texts[idx]),

            "filename":
            str(metadata_filenames[idx]),

            "similarity":
            float(sim)
        })

    return results

In [17]:
#test
query = "Nike Air Zoom Running Shoes"

results = retrieve_metadata_faiss(

    query,

    top_k=5
)

results

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 791ms/step


[{'metadata': 'Nike React Element 87 Sail Light Bone Shoes',
  'filename': 'clean_2019-09-19_10-01-13_UTC.jpg',
  'similarity': 0.9374737739562988},
 {'metadata': 'Nike Air Max 1 Premium Grey Navy Lifestyle Shoes',
  'filename': 'clean_important_1184.jpg',
  'similarity': 0.9302746653556824},
 {'metadata': 'Nike Air Zoom Alphafly NEXT% Dark Grey Green Shoes',
  'filename': 'padded_image_5974.jpg',
  'similarity': 0.8663958311080933},
 {'metadata': 'Nike ACG Mountain Fly Low Brown Tan Trail Shoes',
  'filename': 'clean_important_67.jpg',
  'similarity': 0.8265069723129272},
 {'metadata': 'Nike Air Monarch IV White Grey Cross Training Shoes',
  'filename': 'resized_padded_new_image_1474.jpg',
  'similarity': 0.7816188335418701}]